In [ ]:
import warnings
warnings.filterwarnings("ignore")

import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

pd.set_option("display.max_columns", 50)
pd.set_option("display.float_format", lambda x: f"{x:,.3f}")

# find the files automatically
for root, _, files in os.walk("/kaggle/input"):
    for f in files:
        print(os.path.join(root, f))
        
BASE = "/kaggle/input/datasets/rikdifos/credit-card-approval-prediction"
app = pd.read_csv(f"{BASE}/application_record.csv")
rec = pd.read_csv(f"{BASE}/credit_record.csv")

print("\napplication_record:", app.shape)
print("credit_record:", rec.shape)

print("\n--- application_record.head() ---")
display(app.head())
print("\n--- credit_record.head() ---")
display(rec.head())

print("\n--- application_record.info() ---")
app.info()
print("\n--- credit_record.info() ---")
rec.info()

print("\nUnique applicants in application_record:", app["ID"].nunique())
print("Unique applicants in credit_record:", rec["ID"].nunique())
print("Applicants present in both:", len(set(app["ID"]) & set(rec["ID"])))
print("Duplicate IDs in application_record:", app["ID"].duplicated().sum())

print("\nMissing values in application_record:")
print(app.isna().sum()[app.isna().sum() > 0])

print("\nSTATUS value counts in credit_record:")
print(rec["STATUS"].value_counts())

In [ ]:
# --- duplicates in application_record ---
dups = app[app["ID"].duplicated(keep=False)].sort_values("ID")
print("Rows involved in duplicate IDs:", len(dups))
print("Fully identical duplicate rows:", app.duplicated().sum())
display(dups.head(6))

# --- odd values ---
print("\nDAYS_EMPLOYED describe:")
print(app["DAYS_EMPLOYED"].describe())
print("Rows with DAYS_EMPLOYED > 0 (placeholder?):", (app["DAYS_EMPLOYED"] > 0).sum())
print(app.loc[app["DAYS_EMPLOYED"] > 0, "NAME_INCOME_TYPE"].value_counts())

print("\nConstant / near-constant columns:")
for c in ["FLAG_MOBIL", "FLAG_WORK_PHONE", "FLAG_PHONE", "FLAG_EMAIL"]:
    print(c, app[c].value_counts(normalize=True).round(3).to_dict())

# --- credit_record ---
print("\nSTATUS value counts:")
print(rec["STATUS"].value_counts())
print("\nMONTHS_BALANCE range:", rec["MONTHS_BALANCE"].min(), "to", rec["MONTHS_BALANCE"].max())

# --- target: bad = ever 60+ days overdue (STATUS 2,3,4,5) ---
rec["is_bad"] = rec["STATUS"].isin(["2", "3", "4", "5"]).astype(int)
target = rec.groupby("ID").agg(
    bad_flag=("is_bad", "max"),
    n_months=("MONTHS_BALANCE", "count"),
    history_start=("MONTHS_BALANCE", "min")
).reset_index()

# --- merge (drop duplicate application IDs first) ---
app_unique = app.drop_duplicates(subset="ID", keep="first")
df = app_unique.merge(target, on="ID", how="inner")

print("\nMerged dataset shape:", df.shape)
print("\nTarget distribution (1 = bad):")
print(df["bad_flag"].value_counts())
print(df["bad_flag"].value_counts(normalize=True).round(4))
print("\nMissing OCCUPATION_TYPE in merged data:", df["OCCUPATION_TYPE"].isna().sum(),
      f"({df['OCCUPATION_TYPE'].isna().mean():.1%})")

In [ ]:
# --- drop ambiguous (shared) IDs entirely, then merge ---
dup_ids = app.loc[app["ID"].duplicated(keep=False), "ID"].unique()
app_clean = app[~app["ID"].isin(dup_ids)].copy()
df = app_clean.merge(target, on="ID", how="inner")

print("Shape after dropping shared IDs:", df.shape)
print(df["bad_flag"].value_counts().to_dict(), f"| bad rate: {df['bad_flag'].mean():.2%}")

# --- readable helper columns (raw EDA only) ---
df["AGE_YEARS"] = (-df["DAYS_BIRTH"] / 365.25).round(1)
df["EMPLOYED_YEARS"] = np.where(df["DAYS_EMPLOYED"] == 365243, np.nan, -df["DAYS_EMPLOYED"] / 365.25)

# --- is occupation missingness tied to pensioners? ---
df["OCC_MISSING"] = df["OCCUPATION_TYPE"].isna()
print("\nShare of missing occupation by income type:")
print(df.groupby("NAME_INCOME_TYPE")["OCC_MISSING"].agg(["mean", "sum", "count"]).round(3))

# --- bad rate by category ---
for c in ["NAME_INCOME_TYPE", "NAME_EDUCATION_TYPE", "NAME_FAMILY_STATUS",
          "NAME_HOUSING_TYPE", "CODE_GENDER", "FLAG_OWN_CAR", "FLAG_OWN_REALTY", "OCC_MISSING"]:
    print(f"\nBad rate by {c}:")
    print(df.groupby(c)["bad_flag"].agg(["mean", "sum", "count"]).round(4).sort_values("count", ascending=False))

# --- numeric summary ---
num_cols = ["CNT_CHILDREN", "AMT_INCOME_TOTAL", "CNT_FAM_MEMBERS", "AGE_YEARS", "EMPLOYED_YEARS", "n_months"]
print("\nNumeric summary:")
print(df[num_cols].describe().round(2))

print("\nMean history length (months) by target:")
print(df.groupby("bad_flag")["n_months"].mean().round(2))

In [ ]:
import os
from matplotlib.ticker import PercentFormatter

OUT = "/kaggle/working/figures/eda_raw"
os.makedirs(OUT, exist_ok=True)

sns.set_theme(style="whitegrid", context="notebook")
C_GOOD, C_BAD, C_MAIN = "#3A7CA5", "#D1495B", "#2F4858"
overall = df["bad_flag"].mean()

def save(name):
    plt.tight_layout()
    plt.savefig(f"{OUT}/{name}.png", dpi=200, bbox_inches="tight")
    plt.show()

# 1. target balance
fig, ax = plt.subplots(1, 2, figsize=(11, 4))
vc = df["bad_flag"].map({0: "Good", 1: "Bad"}).value_counts()
ax[0].bar(vc.index, vc.values, color=[C_GOOD, C_BAD])
for i, v in enumerate(vc.values):
    ax[0].text(i, v, f"{v:,}", ha="center", va="bottom")
ax[0].set_title("Target distribution (counts)")
ax[1].pie(vc.values, labels=vc.index, colors=[C_GOOD, C_BAD], autopct="%1.1f%%", startangle=90)
ax[1].set_title("Target distribution (share)")
save("01_target_balance")

# 2. missing values
miss = df.isna().mean().mul(100)
miss = miss[miss > 0].sort_values()
fig, ax = plt.subplots(figsize=(8, 3))
ax.barh(miss.index, miss.values, color=C_MAIN)
for i, v in enumerate(miss.values):
    ax.text(v, i, f" {v:.1f}%", va="center")
ax.set_xlabel("% missing"); ax.set_title("Missing values (raw merged data)")
save("02_missing_values")

# 3. numeric distributions
num_plot = ["AGE_YEARS", "EMPLOYED_YEARS", "AMT_INCOME_TOTAL", "CNT_CHILDREN", "CNT_FAM_MEMBERS"]
fig, axes = plt.subplots(2, 3, figsize=(15, 8))
for ax, c in zip(axes.ravel(), num_plot):
    sns.histplot(df[c].dropna(), bins=40, color=C_MAIN, ax=ax)
    ax.set_title(f"Distribution: {c}")
axes.ravel()[-1].axis("off")
save("03_numeric_distributions")

# 4. boxplots by target
fig, axes = plt.subplots(1, 5, figsize=(18, 4))
for ax, c in zip(axes, num_plot):
    sns.boxplot(data=df, x="bad_flag", y=c, palette=[C_GOOD, C_BAD], ax=ax)
    ax.set_title(c); ax.set_xlabel("bad_flag")
save("04_numeric_by_target")

# 5. bad rate by categorical feature
cat_plot = ["NAME_INCOME_TYPE", "NAME_EDUCATION_TYPE", "NAME_FAMILY_STATUS",
            "NAME_HOUSING_TYPE", "CODE_GENDER", "FLAG_OWN_CAR", "FLAG_OWN_REALTY", "OCCUPATION_TYPE"]
fig, axes = plt.subplots(4, 2, figsize=(15, 18))
for ax, c in zip(axes.ravel(), cat_plot):
    g = df.assign(**{c: df[c].fillna("Unknown")}).groupby(c)["bad_flag"].agg(["mean", "count"]).sort_values("mean")
    ax.barh(g.index, g["mean"], color=C_BAD)
    ax.axvline(overall, color="black", ls="--", lw=1)
    for i, (m, n) in enumerate(zip(g["mean"], g["count"])):
        ax.text(m, i, f" {m:.1%} (n={n:,})", va="center", fontsize=8)
    ax.xaxis.set_major_formatter(PercentFormatter(1.0))
    ax.set_title(f"Bad rate by {c}"); ax.set_xlabel(""); ax.set_ylabel("")
save("05_bad_rate_by_category")

# 6. bad rate by binned numerics
fig, axes = plt.subplots(1, 3, figsize=(17, 4.5))
for ax, (col, q, ttl) in zip(axes, [("AGE_YEARS", 8, "Age"), ("AMT_INCOME_TOTAL", 8, "Income"), ("EMPLOYED_YEARS", 6, "Years employed")]):
    b = pd.qcut(df[col], q=q, duplicates="drop")
    g = df.groupby(b)["bad_flag"].mean()
    ax.plot(range(len(g)), g.values, marker="o", color=C_BAD)
    ax.set_xticks(range(len(g))); ax.set_xticklabels([str(i) for i in g.index], rotation=45, ha="right", fontsize=7)
    ax.axhline(overall, color="black", ls="--", lw=1)
    ax.yaxis.set_major_formatter(PercentFormatter(1.0)); ax.set_title(f"Bad rate by {ttl} (quantile bins)")
save("06_bad_rate_by_numeric_bins")

# 7. correlation heatmap (raw numeric, no leakage columns)
corr_cols = ["CNT_CHILDREN", "AMT_INCOME_TOTAL", "CNT_FAM_MEMBERS", "AGE_YEARS", "EMPLOYED_YEARS",
             "FLAG_WORK_PHONE", "FLAG_PHONE", "FLAG_EMAIL", "bad_flag"]
plt.figure(figsize=(9, 7))
sns.heatmap(df[corr_cols].corr(), annot=True, fmt=".2f", cmap="coolwarm", center=0, square=True)
plt.title("Correlation matrix (raw numeric features)")
save("07_correlation_raw")

# 8. leakage check: history length by target
plt.figure(figsize=(6, 4.5))
sns.boxplot(data=df, x="bad_flag", y="n_months", palette=[C_GOOD, C_BAD])
plt.title("History length by target (excluded from features)")
save("08_history_length_leakage_check")

print("Saved:", sorted(os.listdir(OUT)))

In [ ]:
clean = df.copy()

# 1. drop leakage / constant / id columns (ID kept separately)
ids = clean["ID"].copy()
clean = clean.drop(columns=["ID", "FLAG_MOBIL", "n_months", "history_start",
                            "DAYS_BIRTH", "DAYS_EMPLOYED", "OCC_MISSING"])

# 2. employment features (AGE_YEARS and EMPLOYED_YEARS from the raw EDA step are reused)
clean["IS_EMPLOYED"] = clean["EMPLOYED_YEARS"].notna().astype(int)
clean["EMPLOYED_YEARS"] = clean["EMPLOYED_YEARS"].fillna(0)

# 3. missing occupation -> explicit category
clean["OCCUPATION_TYPE"] = clean["OCCUPATION_TYPE"].fillna("Unknown")

# 4. merge tiny categories
clean["NAME_INCOME_TYPE"] = clean["NAME_INCOME_TYPE"].replace({"Student": "Working"})
clean["NAME_EDUCATION_TYPE"] = clean["NAME_EDUCATION_TYPE"].replace({"Academic degree": "Higher education"})

# 5. cap extreme counts, log-transform income
clean["CNT_CHILDREN"] = clean["CNT_CHILDREN"].clip(upper=4)
clean["CNT_FAM_MEMBERS"] = clean["CNT_FAM_MEMBERS"].clip(upper=6)
clean["LOG_INCOME"] = np.log1p(clean["AMT_INCOME_TOTAL"])
clean = clean.drop(columns=["AMT_INCOME_TOTAL"])

# 6. binary encodings
clean["CODE_GENDER"] = (clean["CODE_GENDER"] == "M").astype(int)
clean["FLAG_OWN_CAR"] = (clean["FLAG_OWN_CAR"] == "Y").astype(int)
clean["FLAG_OWN_REALTY"] = (clean["FLAG_OWN_REALTY"] == "Y").astype(int)
clean = clean.rename(columns={"CODE_GENDER": "IS_MALE"})

# 7. target last
clean = clean[[c for c in clean.columns if c != "bad_flag"] + ["bad_flag"]]

# checks
print("Shape:", clean.shape)
print("Missing values:", int(clean.isna().sum().sum()))
print("Bad rate:", f"{clean['bad_flag'].mean():.2%}")
print("\nDtypes:")
print(clean.dtypes.to_string())
print("\nRemaining categoricals:")
for c in clean.select_dtypes("object").columns:
    print(f"  {c}: {clean[c].nunique()} levels")
print("\nCorrelation with target (numeric):")
print(clean.select_dtypes(exclude="object").corr()["bad_flag"].drop("bad_flag").round(3).sort_values().to_string())

feat_cols = [c for c in clean.columns if c != "bad_flag"]
dup_feat = clean.duplicated(subset=feat_cols, keep=False)
print("\nRows with an identical-feature twin:", int(dup_feat.sum()),
      f"({dup_feat.mean():.1%})")

os.makedirs("/kaggle/working/data", exist_ok=True)
clean.to_csv("/kaggle/working/data/clean_credit.csv", index=False)

In [ ]:
feat_cols = [c for c in clean.columns if c != "bad_flag"]
clean["profile_id"] = clean.groupby(feat_cols).ngroup()

g = clean.groupby("profile_id")["bad_flag"].agg(size="size", n_bad="sum")

print("Rows:", len(clean), "| Unique profiles:", len(g))
print("\nProfile size distribution:")
print(g["size"].describe().round(2).to_string())
print("\nTop profile sizes:")
print(g["size"].value_counts().sort_index().head(8).to_string())

mixed = ((g["n_bad"] > 0) & (g["n_bad"] < g["size"])).sum()
multi = (g["size"] > 1).sum()
print(f"\nProfiles with more than one row: {multi}")
print(f"Of those, mixed labels (some good, some bad): {mixed}")

bad_rows_in_multi = clean[clean["bad_flag"] == 1]["profile_id"].map(g["size"]).gt(1).mean()
print(f"Share of bad rows that have a twin: {bad_rows_in_multi:.1%}")

collapsed_bad = (g["n_bad"] > 0).mean()
print(f"\nIf collapsed to one row per profile (bad if any twin is bad):")
print(f"  rows: {len(g)}, bad profiles: {(g['n_bad'] > 0).sum()}, bad rate: {collapsed_bad:.2%}")


In [ ]:
from sklearn.model_selection import StratifiedGroupKFold

X = clean.drop(columns=["bad_flag"])
y = clean["bad_flag"]
groups = clean["profile_id"]

# 5 folds -> first fold is a ~20% group-disjoint, stratified holdout
sgkf = StratifiedGroupKFold(n_splits=5, shuffle=True, random_state=42)
train_idx, test_idx = next(sgkf.split(X, y, groups))

train_df = clean.iloc[train_idx].reset_index(drop=True)
test_df = clean.iloc[test_idx].reset_index(drop=True)

print(f"Train: {len(train_df):,} rows | {train_df['profile_id'].nunique():,} profiles | "
      f"bad: {train_df['bad_flag'].sum()} ({train_df['bad_flag'].mean():.2%})")
print(f"Test : {len(test_df):,} rows | {test_df['profile_id'].nunique():,} profiles | "
      f"bad: {test_df['bad_flag'].sum()} ({test_df['bad_flag'].mean():.2%})")

overlap = set(train_df["profile_id"]) & set(test_df["profile_id"])
print("Profiles shared between train and test:", len(overlap))

print("\nLargest profile per split (rows):",
      train_df["profile_id"].value_counts().max(), "/", test_df["profile_id"].value_counts().max())

train_df.to_csv("/kaggle/working/data/train.csv", index=False)
test_df.to_csv("/kaggle/working/data/test.csv", index=False)

In [ ]:
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from statsmodels.stats.outliers_influence import variance_inflation_factor

OUT = "/kaggle/working/figures/eda_processed"
os.makedirs(OUT, exist_ok=True)

NUM = ["CNT_CHILDREN", "CNT_FAM_MEMBERS", "AGE_YEARS", "EMPLOYED_YEARS", "LOG_INCOME"]
BIN = ["IS_MALE", "FLAG_OWN_CAR", "FLAG_OWN_REALTY", "FLAG_WORK_PHONE", "FLAG_PHONE", "FLAG_EMAIL", "IS_EMPLOYED"]
CAT = ["NAME_INCOME_TYPE", "NAME_EDUCATION_TYPE", "NAME_FAMILY_STATUS", "NAME_HOUSING_TYPE", "OCCUPATION_TYPE"]

X_train, y_train, g_train = train_df[NUM + BIN + CAT], train_df["bad_flag"], train_df["profile_id"]
X_test, y_test = test_df[NUM + BIN + CAT], test_df["bad_flag"]

preprocessor = ColumnTransformer([
    ("num", StandardScaler(), NUM),
    ("bin", "passthrough", BIN),
    ("cat", OneHotEncoder(drop="first", handle_unknown="ignore", sparse_output=False), CAT),
])

Xp = preprocessor.fit_transform(X_train)
feat_names = [n.split("__", 1)[1] for n in preprocessor.get_feature_names_out()]
Xp_df = pd.DataFrame(Xp, columns=feat_names, index=X_train.index)

print("Processed shape:", Xp_df.shape)
print("Scaled numeric mean (should be ~0):", Xp_df[NUM].mean().round(3).to_dict())
print("Scaled numeric std  (should be ~1):", Xp_df[NUM].std().round(3).to_dict())

# 1. before vs after scaling
fig, axes = plt.subplots(2, 5, figsize=(20, 7))
for i, c in enumerate(NUM):
    sns.histplot(X_train[c], bins=35, color=C_MAIN, ax=axes[0, i]); axes[0, i].set_title(f"Before: {c}")
    sns.histplot(Xp_df[c], bins=35, color=C_GOOD, ax=axes[1, i]); axes[1, i].set_title(f"After: {c}")
save("01_before_after_scaling")

# 2. correlation with target (all processed features)
corr_t = Xp_df.assign(bad_flag=y_train.values).corr()["bad_flag"].drop("bad_flag").sort_values()
top = pd.concat([corr_t.head(10), corr_t.tail(10)])
fig, ax = plt.subplots(figsize=(8, 8))
ax.barh(top.index, top.values, color=[C_GOOD if v < 0 else C_BAD for v in top.values])
ax.axvline(0, color="black", lw=0.8); ax.set_title("Top correlations with target (processed, train)")
save("02_processed_target_correlation")
print("\nTop 5 positive:", corr_t.tail(5).round(3).to_dict())
print("Top 5 negative:", corr_t.head(5).round(3).to_dict())

# 3. correlation among numeric + binary
plt.figure(figsize=(10, 8))
sns.heatmap(Xp_df[NUM + BIN].corr(), annot=True, fmt=".2f", cmap="coolwarm", center=0, square=True, annot_kws={"size": 8})
plt.title("Correlation matrix: processed numeric and binary features (train)")
save("03_processed_correlation")

# 4. multicollinearity (VIF)
Xv = Xp_df.astype(float).assign(const=1.0)
vif = pd.Series([variance_inflation_factor(Xv.values, i) for i in range(Xv.shape[1] - 1)],
                index=Xp_df.columns).sort_values(ascending=False)
fig, ax = plt.subplots(figsize=(8, 7))
v = vif.head(15)[::-1]
ax.barh(v.index, v.values, color=C_MAIN); ax.axvline(5, color=C_BAD, ls="--", lw=1)
ax.set_title("Top 15 VIF (dashed line = 5)")
save("04_vif")
print("\nTop VIF:"); print(vif.head(8).round(2).to_string())

# 5. class balance after split
fig, ax = plt.subplots(1, 2, figsize=(9, 3.5))
for a, (n, yy) in zip(ax, [("Train", y_train), ("Test", y_test)]):
    a.bar(["Good", "Bad"], yy.value_counts().sort_index().values, color=[C_GOOD, C_BAD]); a.set_title(f"{n}: bad {yy.mean():.2%}")
save("05_split_balance")

In [ ]:
NUM = ["CNT_CHILDREN", "AGE_YEARS", "EMPLOYED_YEARS", "LOG_INCOME"]
BIN = ["IS_MALE", "FLAG_OWN_CAR", "FLAG_OWN_REALTY", "FLAG_WORK_PHONE", "FLAG_PHONE", "FLAG_EMAIL"]
CAT = ["NAME_INCOME_TYPE", "NAME_EDUCATION_TYPE", "NAME_FAMILY_STATUS", "NAME_HOUSING_TYPE", "OCCUPATION_TYPE"]
FEATURES = NUM + BIN + CAT

X_train, X_test = train_df[FEATURES], test_df[FEATURES]

preprocessor = ColumnTransformer([
    ("num", StandardScaler(), NUM),
    ("bin", "passthrough", BIN),
    ("cat", OneHotEncoder(drop="first", handle_unknown="ignore", sparse_output=False), CAT),
])

Xp = preprocessor.fit_transform(X_train)
feat_names = [n.split("__", 1)[1] for n in preprocessor.get_feature_names_out()]
Xp_df = pd.DataFrame(Xp, columns=feat_names, index=X_train.index)

Xv = Xp_df.astype(float).assign(const=1.0)
vif = pd.Series([variance_inflation_factor(Xv.values, i) for i in range(Xv.shape[1] - 1)],
                index=Xp_df.columns).sort_values(ascending=False)

print("Processed shape:", Xp_df.shape)
print("Features with VIF > 10:", int((vif > 10).sum()), "| VIF > 5:", int((vif > 5).sum()))
print("\nTop VIF:")
print(vif.head(8).round(2).to_string())

fig, ax = plt.subplots(figsize=(8, 7))
v = vif.head(15)[::-1]
ax.barh(v.index, v.values, color=C_MAIN); ax.axvline(5, color=C_BAD, ls="--", lw=1)
ax.set_title("Top 15 VIF after fixing collinearity (dashed line = 5)")
save("06_vif_after_fix")

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.dummy import DummyClassifier
from sklearn.metrics import make_scorer, f1_score, recall_score, precision_score
from sklearn.model_selection import cross_validate, StratifiedGroupKFold
from imblearn.pipeline import Pipeline as ImbPipeline
from imblearn.over_sampling import SMOTE
from imblearn.under_sampling import RandomUnderSampler

cv = StratifiedGroupKFold(n_splits=5, shuffle=True, random_state=42)

scoring = {
    "roc_auc": "roc_auc",
    "pr_auc": "average_precision",
    "balanced_acc": "balanced_accuracy",
    "recall_bad": make_scorer(recall_score, zero_division=0),
    "precision_bad": make_scorer(precision_score, zero_division=0),
    "f1_bad": make_scorer(f1_score, zero_division=0),
}

def lr(**kw):
    return LogisticRegression(max_iter=3000, random_state=42, **kw)

experiments = {
    "Dummy (prior)":        ImbPipeline([("prep", preprocessor), ("clf", DummyClassifier(strategy="prior"))]),
    "LR plain":             ImbPipeline([("prep", preprocessor), ("clf", lr())]),
    "LR class_weight":      ImbPipeline([("prep", preprocessor), ("clf", lr(class_weight="balanced"))]),
    "LR + SMOTE":           ImbPipeline([("prep", preprocessor), ("smote", SMOTE(random_state=42)), ("clf", lr())]),
    "LR + undersampling":   ImbPipeline([("prep", preprocessor), ("rus", RandomUnderSampler(random_state=42)), ("clf", lr())]),
}

rows = []
for name, pipe in experiments.items():
    res = cross_validate(pipe, X_train, y_train, groups=g_train, cv=cv, scoring=scoring, n_jobs=-1)
    row = {"model": name}
    for m in scoring:
        row[m] = res[f"test_{m}"].mean()
        row[m + "_std"] = res[f"test_{m}"].std()
    rows.append(row)
    print(f"done: {name}")

cv_base = pd.DataFrame(rows).set_index("model")
show = cv_base[[c for c in cv_base.columns if not c.endswith("_std")]].round(4)
print("\nGrouped 5-fold CV (train only), mean scores:")
print(show.to_string())
print("\nStd of ROC-AUC and PR-AUC across folds:")
print(cv_base[["roc_auc_std", "pr_auc_std"]].round(4).to_string())
print("\nPR-AUC of a random classifier = bad rate =", round(y_train.mean(), 4))

os.makedirs("/kaggle/working/results", exist_ok=True)
cv_base.to_csv("/kaggle/working/results/cv_baselines.csv")

In [ ]:
import joblib
from sklearn.model_selection import GridSearchCV

pipe = ImbPipeline([("prep", preprocessor), ("clf", LogisticRegression(solver="liblinear", max_iter=3000, random_state=42))])

param_grid = {
    "clf__C": [0.001, 0.01, 0.03, 0.1, 0.3, 1, 3, 10, 100],
    "clf__penalty": ["l1", "l2"],
    "clf__class_weight": [None, "balanced"],
}

gs = GridSearchCV(
    pipe, param_grid, scoring=scoring, refit="pr_auc",
    cv=StratifiedGroupKFold(n_splits=5, shuffle=True, random_state=7),
    n_jobs=-1, return_train_score=True, verbose=1,
)
gs.fit(X_train, y_train, groups=g_train)

print("Best params:", gs.best_params_)
print("Best CV PR-AUC:", round(gs.best_score_, 4))

res = pd.DataFrame(gs.cv_results_)
cols = ["param_clf__C", "param_clf__penalty", "param_clf__class_weight",
        "mean_test_pr_auc", "mean_test_roc_auc", "mean_test_recall_bad",
        "mean_test_precision_bad", "mean_test_f1_bad", "mean_train_pr_auc", "std_test_pr_auc"]
top = res.sort_values("mean_test_pr_auc", ascending=False)[cols].head(10)
top.columns = ["C", "penalty", "class_weight", "cv_pr_auc", "cv_roc_auc", "cv_recall", "cv_prec", "cv_f1", "train_pr_auc", "pr_auc_std"]
print("\nTop 10 configurations:")
print(top.round(4).to_string(index=False))

best_row = res.loc[gs.best_index_]
print("\nBest model, all CV metrics:")
for m in scoring:
    print(f"  {m}: {best_row['mean_test_' + m]:.4f} (+/- {best_row['std_test_' + m]:.4f})")

res.to_csv("/kaggle/working/results/gridsearch_results.csv", index=False)
os.makedirs("/kaggle/working/models", exist_ok=True)
joblib.dump(gs.best_estimator_, "/kaggle/working/models/lr_best.joblib")

In [ ]:
from sklearn.base import clone
from sklearn.model_selection import cross_val_predict
from sklearn.metrics import (roc_auc_score, average_precision_score, precision_recall_curve,
                             roc_curve, confusion_matrix, brier_score_loss, balanced_accuracy_score)
from sklearn.calibration import calibration_curve

OUT = "/kaggle/working/figures/model"
os.makedirs(OUT, exist_ok=True)
best = gs.best_estimator_

# 1. out-of-fold train probabilities (group-aware) -> choose thresholds
oof = cross_val_predict(clone(best), X_train, y_train, groups=g_train,
                        cv=StratifiedGroupKFold(n_splits=5, shuffle=True, random_state=11),
                        method="predict_proba", n_jobs=-1)[:, 1]
prec, rec, thr = precision_recall_curve(y_train, oof)
f1 = 2 * prec * rec / (prec + rec + 1e-12)
thr_f1 = thr[np.argmax(f1[:-1])]
thr_r50 = thr[np.where(rec[:-1] >= 0.50)[0].max()]
print(f"OOF train ROC-AUC {roc_auc_score(y_train, oof):.4f} | PR-AUC {average_precision_score(y_train, oof):.4f}")
print(f"Threshold (max F1): {thr_f1:.4f} | Threshold (recall >= 50%): {thr_r50:.4f}")

# 2. one-shot test evaluation
p_test = best.predict_proba(X_test)[:, 1]
print(f"\nTEST ROC-AUC {roc_auc_score(y_test, p_test):.4f} | PR-AUC {average_precision_score(y_test, p_test):.4f} "
      f"| Brier {brier_score_loss(y_test, p_test):.4f} (always-base-rate Brier {y_train.mean()*(1-y_train.mean()):.4f})")

def report(name, t):
    pred = (p_test >= t).astype(int)
    tn, fp, fn, tp = confusion_matrix(y_test, pred).ravel()
    pr = tp / max(tp + fp, 1); rc = tp / (tp + fn)
    print(f"{name:<22} thr={t:.4f} | flagged {pred.mean():6.1%} | TP {tp:3d} FN {fn:3d} FP {fp:4d} TN {tn:4d} "
          f"| prec {pr:.3f} rec {rc:.3f} F1 {2*pr*rc/(pr+rc+1e-12):.3f} bal_acc {balanced_accuracy_score(y_test, pred):.3f}")
    return pred, (tn, fp, fn, tp)

print("\nTest results at chosen thresholds:")
report("Default 0.5", 0.5)
pred_f1, cm_f1 = report("Max-F1 threshold", thr_f1)
pred_r50, cm_r50 = report("Recall>=50% threshold", thr_r50)

# 3. grouped bootstrap CIs (resample profiles, not rows)
gid = test_df["profile_id"].values
idx_by_g = {k: v.values for k, v in pd.Series(np.arange(len(gid))).groupby(gid)}
ug = list(idx_by_g); rng = np.random.default_rng(42); yt = y_test.values
boot = []
for _ in range(500):
    s = rng.choice(len(ug), len(ug))
    ix = np.concatenate([idx_by_g[ug[i]] for i in s])
    if yt[ix].sum() == 0: continue
    boot.append((roc_auc_score(yt[ix], p_test[ix]), average_precision_score(yt[ix], p_test[ix])))
boot = np.array(boot)
print("\nGrouped bootstrap 95% CI on test:")
print(f"  ROC-AUC: {np.percentile(boot[:,0], 2.5):.3f} to {np.percentile(boot[:,0], 97.5):.3f}")
print(f"  PR-AUC : {np.percentile(boot[:,1], 2.5):.3f} to {np.percentile(boot[:,1], 97.5):.3f}  (random = {y_test.mean():.3f})")

# 4. decile lift on test
dec = pd.qcut(pd.Series(p_test).rank(method="first"), 10, labels=False)
lift = pd.DataFrame({"decile": dec, "bad": yt}).groupby("decile")["bad"].agg(["mean", "sum", "count"])
lift["lift"] = lift["mean"] / yt.mean()
lift.index = [f"D{i+1}" for i in lift.index]
print("\nDecile lift on test (D10 = highest predicted risk):")
print(lift.round(3).to_string())

# 5. coefficients as odds ratios
names = [n.split("__", 1)[1] for n in best.named_steps["prep"].get_feature_names_out()]
coef = pd.Series(best.named_steps["clf"].coef_[0], index=names)
orr = np.exp(coef).sort_values()
print("\nLowest odds ratios (protective):"); print(orr.head(5).round(3).to_string())
print("Highest odds ratios (riskier):"); print(orr.tail(5).round(3).to_string())

# figures
fig, ax = plt.subplots(1, 3, figsize=(17, 4.8))
fr, tr, _ = roc_curve(y_test, p_test); fo, to, _ = roc_curve(y_train, oof)
ax[0].plot(fo, to, color=C_GOOD, label=f"Train OOF ({roc_auc_score(y_train, oof):.3f})")
ax[0].plot(fr, tr, color=C_BAD, label=f"Test ({roc_auc_score(y_test, p_test):.3f})")
ax[0].plot([0, 1], [0, 1], "k--", lw=1); ax[0].set_title("ROC curve"); ax[0].legend()
pt, rt, _ = precision_recall_curve(y_test, p_test)
ax[1].plot(rt, pt, color=C_BAD); ax[1].axhline(y_test.mean(), color="black", ls="--", lw=1)
ax[1].set_title(f"Precision-Recall (test, PR-AUC {average_precision_score(y_test, p_test):.3f})")
fp_, mp_ = calibration_curve(y_test, p_test, n_bins=8, strategy="quantile")
ax[2].plot(mp_, fp_, marker="o", color=C_MAIN); ax[2].plot([0, 0.06], [0, 0.06], "k--", lw=1)
ax[2].set_title("Calibration (test)"); ax[2].set_xlabel("Predicted"); ax[2].set_ylabel("Observed")
save("01_roc_pr_calibration")

fig, ax = plt.subplots(1, 3, figsize=(17, 4.5))
for a, (t, cm) in zip(ax[:2], [("Max-F1 threshold", cm_f1), ("Recall>=50% threshold", cm_r50)]):
    tn, fp, fn, tp = cm
    sns.heatmap([[tn, fp], [fn, tp]], annot=True, fmt="d", cmap="Blues", cbar=False, ax=a,
                xticklabels=["Pred good", "Pred bad"], yticklabels=["Actual good", "Actual bad"])
    a.set_title(f"Confusion matrix: {t}")
ax[2].bar(lift.index, lift["lift"], color=C_MAIN); ax[2].axhline(1, color="black", ls="--", lw=1)
ax[2].set_title("Decile lift (test)")
save("02_confusion_and_lift")

sel = pd.concat([coef.sort_values().head(10), coef.sort_values().tail(10)])
fig, ax = plt.subplots(figsize=(8, 8))
ax.barh(sel.index, sel.values, color=[C_GOOD if v < 0 else C_BAD for v in sel.values])
ax.axvline(0, color="black", lw=0.8); ax.set_title("Largest logistic coefficients (standardized features)")
save("03_coefficients")

pd.DataFrame({"coef": coef, "odds_ratio": np.exp(coef)}).sort_values("coef").to_csv("/kaggle/working/results/lr_coefficients.csv")
pd.DataFrame({"y_true": yt, "p_bad": p_test}).to_csv("/kaggle/working/results/test_predictions.csv", index=False)

In [ ]:
from sklearn.ensemble import RandomForestClassifier, HistGradientBoostingClassifier

models = {
    "Dummy (prior)": DummyClassifier(strategy="prior"),
    "Logistic Regression (tuned)": clone(best.named_steps["clf"]),
    "Random Forest": RandomForestClassifier(n_estimators=300, max_depth=8, min_samples_leaf=50,
                                            class_weight="balanced_subsample", n_jobs=-1, random_state=42),
    "HistGradientBoosting": HistGradientBoostingClassifier(max_depth=3, learning_rate=0.05, max_iter=150,
                                                           min_samples_leaf=100, l2_regularization=5.0,
                                                           random_state=42),
}

cv_cmp = StratifiedGroupKFold(n_splits=5, shuffle=True, random_state=21)
rows = []
for name, clf in models.items():
    pipe_m = ImbPipeline([("prep", preprocessor), ("clf", clf)])
    r = cross_validate(pipe_m, X_train, y_train, groups=g_train, cv=cv_cmp,
                       scoring={"roc_auc": "roc_auc", "pr_auc": "average_precision"}, n_jobs=-1)
    pipe_m.fit(X_train, y_train)
    pt = pipe_m.predict_proba(X_test)[:, 1]
    rows.append({
        "model": name,
        "cv_roc_auc": r["test_roc_auc"].mean(), "cv_roc_std": r["test_roc_auc"].std(),
        "cv_pr_auc": r["test_pr_auc"].mean(), "cv_pr_std": r["test_pr_auc"].std(),
        "test_roc_auc": roc_auc_score(y_test, pt), "test_pr_auc": average_precision_score(y_test, pt),
    })
    print("done:", name)

cmp_df = pd.DataFrame(rows).set_index("model")
print("\nModel comparison (grouped CV on train, one-shot test):")
print(cmp_df.round(4).to_string())
print(f"\nRandom-classifier PR-AUC: train {y_train.mean():.4f} | test {y_test.mean():.4f}")

fig, ax = plt.subplots(1, 2, figsize=(13, 4.5))
x = np.arange(len(cmp_df))
ax[0].bar(x, cmp_df["cv_roc_auc"], yerr=cmp_df["cv_roc_std"], color=C_MAIN, capsize=4, label="CV")
ax[0].bar(x + 0.35, cmp_df["test_roc_auc"], width=0.35, color=C_BAD, label="Test")
ax[0].axhline(0.5, color="black", ls="--", lw=1); ax[0].set_ylim(0.4, 0.7)
ax[0].set_xticks(x + 0.17); ax[0].set_xticklabels(cmp_df.index, rotation=20, ha="right"); ax[0].set_title("ROC-AUC"); ax[0].legend()
ax[1].bar(x, cmp_df["cv_pr_auc"], yerr=cmp_df["cv_pr_std"], color=C_MAIN, capsize=4, label="CV")
ax[1].bar(x + 0.35, cmp_df["test_pr_auc"], width=0.35, color=C_BAD, label="Test")
ax[1].axhline(y_train.mean(), color="black", ls="--", lw=1)
ax[1].set_xticks(x + 0.17); ax[1].set_xticklabels(cmp_df.index, rotation=20, ha="right"); ax[1].set_title("PR-AUC (dashed = random)"); ax[1].legend()
save("04_model_comparison")

cmp_df.to_csv("/kaggle/working/results/model_comparison.csv")

In [ ]:
credit_hist = pd.read_csv(f"{BASE}/credit_record.csv")

defs = {
    "30+ days late (status 1-5)": ["1", "2", "3", "4", "5"],
    "60+ days late (current)":    ["2", "3", "4", "5"],
    "90+ days late (status 3-5)": ["3", "4", "5"],
}

rows = []
for dname, codes in defs.items():
    lab = credit_hist.assign(b=credit_hist["STATUS"].isin(codes).astype(int)).groupby("ID")["b"].max()
    y_all = ids.map(lab).astype(int)
    ytr, yte = y_all.iloc[train_idx].reset_index(drop=True), y_all.iloc[test_idx].reset_index(drop=True)
    for mname, clf in [("LR", LogisticRegression(C=0.3, max_iter=3000, random_state=42)),
                       ("HGB", HistGradientBoostingClassifier(max_depth=3, learning_rate=0.05, max_iter=150,
                                                              min_samples_leaf=100, l2_regularization=5.0, random_state=42))]:
        pipe_s = ImbPipeline([("prep", preprocessor), ("clf", clf)])
        r = cross_validate(pipe_s, X_train, ytr, groups=g_train,
                           cv=StratifiedGroupKFold(n_splits=5, shuffle=True, random_state=21),
                           scoring={"roc_auc": "roc_auc", "pr_auc": "average_precision"}, n_jobs=-1)
        pipe_s.fit(X_train, ytr)
        pt = pipe_s.predict_proba(X_test)[:, 1]
        rows.append({"label": dname, "model": mname, "bad_train": int(ytr.sum()), "bad_rate": ytr.mean(),
                     "cv_roc": r["test_roc_auc"].mean(), "cv_roc_std": r["test_roc_auc"].std(),
                     "cv_pr": r["test_pr_auc"].mean(), "test_roc": roc_auc_score(yte, pt),
                     "test_pr": average_precision_score(yte, pt)})
    print("done:", dname)

sens = pd.DataFrame(rows).set_index(["label", "model"])
print("\nLabel-definition sensitivity:")
print(sens.round(4).to_string())
sens.to_csv("/kaggle/working/results/label_sensitivity.csv")

In [ ]:
import json, shutil

final_model = joblib.load("/kaggle/working/models/lr_best.joblib")
THRESHOLD = float(thr_f1)   # chosen on out-of-fold train predictions

def predict_approval(applicant: dict, threshold: float = THRESHOLD) -> dict:
    """applicant: dict with the 15 raw feature columns used in training (AGE_YEARS, EMPLOYED_YEARS, LOG_INCOME, ...)"""
    row = pd.DataFrame([applicant])[FEATURES]
    p_bad = float(final_model.predict_proba(row)[0, 1])
    return {
        "risk_probability": round(p_bad, 4),
        "base_rate": round(float(y_train.mean()), 4),
        "decision": "REVIEW / DECLINE" if p_bad >= threshold else "APPROVE",
    }

# demo on 3 random test applicants
demo = X_test.sample(3, random_state=1)
for i, (_, r) in enumerate(demo.iterrows(), 1):
    print(f"Applicant {i}:", predict_approval(r.to_dict()))

# final summary
summary = {
    "dataset": "rikdifos/credit-card-approval-prediction",
    "modeling_rows": int(len(clean)), "unique_profiles": int(clean["profile_id"].nunique()),
    "target": "bad = ever 60+ days overdue", "bad_rate_rows": round(float(clean["bad_flag"].mean()), 4),
    "split": "StratifiedGroupKFold by identical-feature profile (no leakage)",
    "train_rows": int(len(train_df)), "test_rows": int(len(test_df)),
    "best_params": {k: (v if v is not None else "None") for k, v in gs.best_params_.items()},
    "cv_roc_auc": round(float(best_row["mean_test_roc_auc"]), 4),
    "cv_pr_auc": round(float(best_row["mean_test_pr_auc"]), 4),
    "test_roc_auc": round(float(roc_auc_score(y_test, p_test)), 4),
    "test_pr_auc": round(float(average_precision_score(y_test, p_test)), 4),
    "test_roc_auc_95ci": [round(float(np.percentile(boot[:, 0], 2.5)), 3), round(float(np.percentile(boot[:, 0], 97.5)), 3)],
    "decision_threshold": round(THRESHOLD, 4),
}
with open("/kaggle/working/results/final_summary.json", "w") as f:
    json.dump(summary, f, indent=2)
print("\nFinal summary:"); print(json.dumps(summary, indent=2))

# small data sample for the repo (full CSVs stay out of GitHub)
os.makedirs("/kaggle/working/data_sample", exist_ok=True)
clean.drop(columns=["profile_id"]).head(200).to_csv("/kaggle/working/data_sample/clean_sample.csv", index=False)

# one zip with everything to download
shutil.make_archive("/kaggle/working/credit_approval_outputs", "zip", "/kaggle/working",
                    base_dir=None) if False else None
for folder in ["figures", "results", "models", "data_sample"]:
    shutil.make_archive(f"/kaggle/working/zip_{folder}", "zip", f"/kaggle/working/{folder}")
print("\nFiles ready:", sorted(f for f in os.listdir("/kaggle/working") if f.endswith(".zip")))
print("Figures:", {d: len(os.listdir(f'/kaggle/working/figures/{d}')) for d in os.listdir('/kaggle/working/figures')})